# Estatísticas do artigo

SVM · `l1_stable` · nocombat. ΔAUC pareado (bootstrap) + FDR BH.
Claim: **`48m_6m`**. Contrastes: **Q4 `t1_d21_d32` vs T1** e **D21 `t1_d21` vs T1**.
Figs: `6_results.ipynb`. Sem `abs` / `t1_deltas` / `wide`.

| § | Pergunta | Ficheiro |
|---|----------|----------|
| 1 | Demografia sMCI vs pMCI (`48m_6m`) | `stats_demo_48m6m` |
| 2 | Idade/sexo confundem imagem? (shape T1) | `stats_confound_48m6m` |
| **3** | **Q4 vs T1 e D21 vs T1, 5 famílias, FDR** | **`stats_q4_vs_t1_48m6m` / `stats_d21_vs_t1_48m6m`** |
| 4 | Mesmos contrastes × 4 coortes (células sobrepõem) | `stats_*_vs_t1_gradient` |
| 5 | Late vs teto shape T1 (3 specs) | `stats_late_vs_shape_48m6m` |
| 6 | Clínico vs vol T1 | `stats_clinic_48m6m` |
| 7 | Vol Q4 vs leaky | `stats_leaky_48m6m` |
| 8 | Soft True vs False (descritivo) | `stats_soft_true_vs_false` |
| 9 | ComBat vs nocombat (sensibilidade) | `stats_combat_vs_nocombat` |
| 10 | Tabela D — resumo complementares | `stats_table_d_complements` |


In [ ]:
from __future__ import annotations

import sys
from dataclasses import dataclass
from pathlib import Path

_MOD = Path.cwd() / "modules"
if str(_MOD) not in sys.path:
    sys.path.insert(0, str(_MOD))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from ablation_analysis import explode_patient_predictions, prepare_ablation_df
from stats_compare import (
    apply_bh_fdr,
    clinical_results_path,
    compare_modalities,
    fusion_results_path,
    image_ablation_path,
    paired_comparison_row,
    print_comparison_summary,
)

COHORT_CLAIM = "48m_6m"
COHORTS_GRADIENT = ("36m_6m", "36m_12m", "48m_6m", "48m_12m")
BASE = Path(f"csvs/cohorts/{COHORT_CLAIM}")
LONG_CSV = BASE / "adnimerged_longitudinal.csv"
TABLE_DIR = Path("Artigo 1 pgirardi") / "tables"

PROTOCOL_BASELINE = "t1_only"
PROTOCOL_TWO_VISIT = "t1_d21"  # D21
PROTOCOL_LONGITUDINAL = "t1_d21_d32"  # Q4
MODS_COMPARE = ("vol", "shape", "texture", "disp", "firstorder")
CLINIC_MODALITY = "vol"
SOFT_FALSE_COHORT = "48m_6m_soft_False"
ALPHA = 0.05

LATE_ALL_T1 = "late__t1_vol__t1_shape__t1_texture__t1_disp__t1_firstorder"
LATE_ALL_Q4 = (
    "late__t1_d21d32_vol__t1_d21d32_shape__t1_d21d32_texture"
    "__t1_d21d32_disp__t1_d21d32_firstorder"
)
LATE_ANCORA = (
    "late__t1_shape__t1_d21d32_vol__t1_d21d32_texture"
    "__t1_d21d32_disp__t1_d21d32_firstorder"
)

FOLDER_REP = {
    "ablation_results_t1_only": "t1_only",
    "ablation_results_d21": "t1_d21",
    "ablation_results_d21d32": "t1_d21_d32",
    "ablation_results_leaky_d21d32": "t1_d21_d32",
    "ablation_results_combat_t1_only": "t1_only",
    "ablation_results_combat_d21d32": "t1_d21_d32",
    "ablation_results_clinic_img_t1_only": "t1_only",
    "ablation_results_late_fusion": "late_fusion",
}


@dataclass(frozen=True)
class StatsCfg:
    task: str = "smci_pmci"
    groups: tuple[str, str] = ("sMCI", "pMCI")
    positive_group: str = "pMCI"
    modality: str = "shape"
    model_key: str = "svm"
    with_combat: bool = False
    selection_mode: str = "l1_stable"
    protocol: str = "t1_only"
    n_perm: int = 5000
    n_bootstrap: int = 5000
    seed: int = 42


STATS_CFG = StatsCfg()


def ler_p(p: float, *, contexto: str = "") -> str:
    tag = "significativo" if p < ALPHA else "não significativo"
    prefix = f"{contexto}: " if contexto else ""
    return f"{prefix}{tag} (p={p:.4f})"


def save_table(df: pd.DataFrame, name: str) -> None:
    TABLE_DIR.mkdir(parents=True, exist_ok=True)
    path = TABLE_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    print("Salvo:", path)


print("claim:", BASE, "|", PROTOCOL_LONGITUDINAL, "&", PROTOCOL_TWO_VISIT, "vs", PROTOCOL_BASELINE)
print("mods:", MODS_COMPARE, "| model:", STATS_CFG.model_key, STATS_CFG.selection_mode)
print("clinic+img:", CLINIC_MODALITY, "| soft_false:", SOFT_FALSE_COHORT)
print("perm/boot:", STATS_CFG.n_perm, STATS_CFG.n_bootstrap)


claim: csvs/cohorts/48m_6m | t1_d21_d32 & t1_d21 vs t1_only
mods: ('vol', 'shape', 'texture', 'disp', 'firstorder') | model: svm l1_stable
clinic+img: vol | soft_false: 48m_6m_soft_False
perm/boot: 5000 5000


In [11]:
def load_cohort(long_path: Path, cfg: StatsCfg) -> pd.DataFrame:
    sub = pd.read_csv(long_path)
    sub = sub[sub["slot"].astype(str).isin(("baseline", "t0"))]
    sub = sub.sort_values(["ID_PT", "ID_IMG"]).groupby("ID_PT", as_index=False).first()
    sub = sub[sub["GROUP"].isin(cfg.groups)].copy()
    sub["ID_PT"] = sub["ID_PT"].astype(str)
    sub["y"] = (sub["GROUP"] == cfg.positive_group).astype(int)
    sub["SEX_bin"] = sub["SEX"].map({"M": 0, "F": 1, 0: 0, 1: 1})
    return sub


def permutation_auc_p(
    y, scores, *, n_perm: int, seed: int, bidirectional: bool = False,
) -> tuple[float, float]:
    y = np.asarray(y, dtype=int)
    s = np.asarray(scores, dtype=float)
    auc_obs = float(
        max(roc_auc_score(y, s), roc_auc_score(y, -s)) if bidirectional
        else roc_auc_score(y, s)
    )
    prng = np.random.default_rng(seed)
    ge = 0
    for _ in range(n_perm):
        yp = prng.permutation(y)
        auc_n = (
            max(roc_auc_score(yp, s), roc_auc_score(yp, -s)) if bidirectional
            else roc_auc_score(yp, s)
        )
        ge += int(auc_n >= auc_obs)
    return auc_obs, (ge + 1) / (n_perm + 1)


def bootstrap_auc_diff(y, scores_a, scores_b, *, n_boot: int, seed: int):
    y = np.asarray(y, dtype=int)
    a, b = np.asarray(scores_a, float), np.asarray(scores_b, float)
    obs = float(roc_auc_score(y, a) - roc_auc_score(y, b))
    prng = np.random.default_rng(seed)
    diffs = []
    for _ in range(n_boot):
        idx = prng.integers(0, len(y), size=len(y))
        yb, ab, bb = y[idx], a[idx], b[idx]
        if len(np.unique(yb)) < 2:
            continue
        diffs.append(float(roc_auc_score(yb, ab) - roc_auc_score(yb, bb)))
    diffs = np.asarray(diffs)
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return obs, float(lo), float(hi)


def nested_cv_auc_univariate(X, y, *, seed: int, k: int = 5):
    X = np.asarray(X, dtype=float).reshape(-1, 1)
    y = np.asarray(y, dtype=int)
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=5000, random_state=seed)),
    ])
    cv = StratifiedKFold(k, shuffle=True, random_state=seed)
    scores = cross_val_predict(pipe, X, y, cv=cv, method="predict_proba")[:, 1]
    return float(roc_auc_score(y, scores)), scores


def patient_image_scores(
    path: Path, cfg: StatsCfg, *, expect_representation: str | None = None,
) -> pd.DataFrame:
    raw = prepare_ablation_df(pd.read_csv(path))
    mask = (
        (raw["task"] == cfg.task)
        & (raw["modality"] == cfg.modality)
        & (raw["model_key"] == cfg.model_key)
        & (raw["with_combat"] == cfg.with_combat)
        & (raw["selection_mode"] == cfg.selection_mode)
    )
    if expect_representation is not None and "representation" in raw.columns:
        mask = mask & (raw["representation"].astype(str) == expect_representation)
    sub = raw.loc[mask]
    if sub.empty:
        raise FileNotFoundError(f"sem linhas para {cfg} rep={expect_representation} em {path}")
    pat = explode_patient_predictions(sub)
    return pat.groupby("ID_PT", as_index=False).agg(
        y=("y", "first"), score_img=("score", "mean"),
    )


def patient_scores_from_path(path: Path, cfg: StatsCfg) -> pd.DataFrame:
    expect = None
    names = {p.name for p in path.parents} | {path.parent.name}
    for folder, rep in FOLDER_REP.items():
        if folder in names:
            expect = rep
            break
    return patient_image_scores(
        path, cfg, expect_representation=expect,
    ).rename(columns={"score_img": "score"})


def cfg_for_modality(mod: str, cfg: StatsCfg | None = None, **kw) -> StatsCfg:
    c = cfg or STATS_CFG
    d = {f.name: getattr(c, f.name) for f in StatsCfg.__dataclass_fields__.values()}
    d["modality"] = mod
    d.update(kw)
    return StatsCfg(**d)


def cfg_clinical(cfg: StatsCfg | None = None) -> StatsCfg:
    return cfg_for_modality("clinical", cfg, with_combat=False, selection_mode="none", protocol="clinical")


def compare_encoding_vs_t1(
    base: Path,
    protocol_long: str,
    *,
    label_a: str,
    cfg: StatsCfg | None = None,
    comparison: str | None = None,
) -> pd.DataFrame:
    c = cfg or STATS_CFG
    tag = comparison or f"{base.name}_{protocol_long}_vs_t1_only"
    return compare_modalities(
        MODS_COMPARE,
        path_a=lambda m: image_ablation_path(base, protocol_long, m),
        path_b=lambda m: image_ablation_path(base, PROTOCOL_BASELINE, m),
        cfg_for_mod=lambda m: cfg_for_modality(m, c),
        load_patients=patient_scores_from_path,
        permutation_auc_p=permutation_auc_p,
        n_perm=c.n_perm,
        n_bootstrap=c.n_bootstrap,
        seed=c.seed,
        label_a=label_a,
        label_b="t1_only",
        comparison=tag,
        alpha=ALPHA,
    )


def compare_q4_vs_t1(
    base: Path,
    cfg: StatsCfg | None = None,
    *,
    comparison: str | None = None,
) -> pd.DataFrame:
    return compare_encoding_vs_t1(
        base,
        PROTOCOL_LONGITUDINAL,
        label_a="q4",
        cfg=cfg,
        comparison=comparison or f"{base.name}_t1_d21_d32_vs_t1_only",
    )


def compare_d21_vs_t1(
    base: Path,
    cfg: StatsCfg | None = None,
    *,
    comparison: str | None = None,
) -> pd.DataFrame:
    return compare_encoding_vs_t1(
        base,
        PROTOCOL_TWO_VISIT,
        label_a="d21",
        cfg=cfg,
        comparison=comparison or f"{base.name}_t1_d21_vs_t1_only",
    )


def combat_ablation_path(base: Path, protocol: str, modality: str) -> Path:
    roots = {
        "t1_only": base / "ablation_results_combat_t1_only",
        "t1_d21_d32": base / "ablation_results_combat_d21d32",
    }
    if protocol not in roots:
        raise KeyError(f"protocolo combat desconhecido: {protocol}")
    return roots[protocol] / modality / "ablation_results_all.csv"


def patient_auc(path: Path, cfg: StatsCfg) -> tuple[float, int]:
    pat = patient_scores_from_path(path, cfg)
    return float(roc_auc_score(pat["y"], pat["score"])), int(len(pat))


cohort = load_cohort(LONG_CSV, STATS_CFG)
g0, g1 = STATS_CFG.groups
print(f"{BASE.name}: n={len(cohort)} | {cohort['GROUP'].value_counts().to_dict()}")


48m_6m: n=193 | {'pMCI': 120, 'sMCI': 73}


## 1. Demografia — `48m_6m`

Idade (Mann-Whitney) e sexo (χ²) entre sMCI e pMCI.


In [12]:
age_a = cohort.loc[cohort["GROUP"] == g0, "AGE"].dropna()
age_b = cohort.loc[cohort["GROUP"] == g1, "AGE"].dropna()
_, p_age_groups = stats.mannwhitneyu(age_a, age_b, alternative="two-sided")
sex_tab = pd.crosstab(
    cohort["GROUP"], cohort["SEX_bin"], rownames=["GROUP"], colnames=["SEX (0=M, 1=F)"],
)
_, p_sex_groups, _, _ = stats.chi2_contingency(sex_tab)
demo = pd.DataFrame([
    {
        "variavel": "idade", "teste": "Mann-Whitney U",
        "media_sMCI": float(age_a.mean()), "media_pMCI": float(age_b.mean()),
        "p": p_age_groups,
    },
    {
        "variavel": "sexo", "teste": "chi2",
        "prop_F_sMCI": float(cohort.loc[cohort["GROUP"] == g0, "SEX_bin"].mean()),
        "prop_F_pMCI": float(cohort.loc[cohort["GROUP"] == g1, "SEX_bin"].mean()),
        "p": p_sex_groups,
    },
])
save_table(demo, "stats_demo_48m6m")
display(sex_tab)
display(demo.round(4))
print(ler_p(p_age_groups, contexto="Idade"))
print(ler_p(p_sex_groups, contexto="Sexo"))


Salvo: artigo/tables/stats_demo_48m6m.csv


"SEX (0=M, 1=F)",0,1
GROUP,,
pMCI,62,58
sMCI,48,25


,variavel,teste,media_sMCI,media_pMCI,p,prop_F_sMCI,prop_F_pMCI
0,idade,Mann-Whitney U,72.2877,74.35,0.0624,NaN,NaN
1,sexo,chi2,NaN,NaN,0.0772,0.3425,0.4833


Idade: não significativo (p=0.0624)
Sexo: não significativo (p=0.0772)


## 2. Idade/sexo vs imagem (shape T1)

Nested CV univariado + permutação. ΔAUC bootstrap: shape T1 − demografia.


In [13]:
# cfg = STATS_CFG
# pt = cohort[["ID_PT", "y", "AGE", "SEX_bin"]].dropna().copy()
# y = pt["y"].to_numpy()
# auc_age, scores_age = nested_cv_auc_univariate(pt["AGE"].to_numpy(), y, seed=cfg.seed)
# _, p_age = permutation_auc_p(y, scores_age, n_perm=cfg.n_perm, seed=cfg.seed)
# auc_sex, scores_sex = nested_cv_auc_univariate(pt["SEX_bin"].to_numpy(), y, seed=cfg.seed + 1)
# _, p_sex = permutation_auc_p(y, scores_sex, n_perm=cfg.n_perm, seed=cfg.seed + 1)
# print(f"idade AUC={auc_age:.3f}  {ler_p(p_age)}")
# print(f"sexo  AUC={auc_sex:.3f}  {ler_p(p_sex)}")

# pt = pt.assign(score_age=scores_age, score_sex=scores_sex)
# img_path = image_ablation_path(BASE, PROTOCOL_BASELINE, cfg.modality)
# img = patient_image_scores(img_path, cfg, expect_representation="t1_only").merge(
#     pt, on=["ID_PT", "y"], how="inner",
# )
# y_m = img["y"].to_numpy()
# auc_img, p_img = permutation_auc_p(
#     y_m, img["score_img"].to_numpy(), n_perm=cfg.n_perm, seed=cfg.seed + 2,
# )
# d_age, lo_age, hi_age = bootstrap_auc_diff(
#     y_m, img["score_img"].to_numpy(), img["score_age"].to_numpy(),
#     n_boot=cfg.n_bootstrap, seed=cfg.seed + 3,
# )
# d_sex, lo_sex, hi_sex = bootstrap_auc_diff(
#     y_m, img["score_img"].to_numpy(), img["score_sex"].to_numpy(),
#     n_boot=cfg.n_bootstrap, seed=cfg.seed + 4,
# )
# confound = pd.DataFrame([
#     {"modelo": "idade", "auc": auc_age, "p_perm": p_age, "delta_vs_img": np.nan, "ci95_lo": np.nan, "ci95_hi": np.nan},
#     {"modelo": "sexo", "auc": auc_sex, "p_perm": p_sex, "delta_vs_img": np.nan, "ci95_lo": np.nan, "ci95_hi": np.nan},
#     {"modelo": "shape T1", "auc": auc_img, "p_perm": p_img, "delta_vs_img": np.nan, "ci95_lo": np.nan, "ci95_hi": np.nan},
#     {"modelo": "shape T1 − idade", "auc": d_age, "p_perm": np.nan, "delta_vs_img": d_age, "ci95_lo": lo_age, "ci95_hi": hi_age},
#     {"modelo": "shape T1 − sexo", "auc": d_sex, "p_perm": np.nan, "delta_vs_img": d_sex, "ci95_lo": lo_sex, "ci95_hi": hi_sex},
# ])
# save_table(confound, "stats_confound_48m6m")
# display(confound.round(4))
# print(f"shape T1 AUC={auc_img:.3f} n={len(img)}  {ler_p(p_img)}")
# print(f"Δ vs idade {d_age:.3f} [{lo_age:.3f}, {hi_age:.3f}]")
# print(f"Δ vs sexo  {d_sex:.3f} [{lo_sex:.3f}, {hi_sex:.3f}]")


# §2 — idade / sexo no protocolo do claim (SVM nested 5×10, Optuna 10)
# selection_mode=none: 1 coluna, sem ℓ1. Mesmos knobs que 5_clinic_img.py.

import importlib.util
import json


from ablation_analysis import patient_mean_auc, patient_mean_predictions
from ablation_optuna import tune_pipeline
from ablation_runner import (
    TASKS,
    fold_metrics,
    gridsearch_n_jobs,
    patient_labels_from_long,
    repeat_ids,
    tune_youden_threshold,
)

_spec = importlib.util.spec_from_file_location("clinic_img", Path.cwd() / "5_clinic_img.py")
clinic = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(clinic)

N_REPEATS = 10          # 1 = smoke
OPTUNA_TRIALS = 10
MODEL_KEY = "svm"
cfg = STATS_CFG

MERGE_LONG = BASE / "ablation" / "hippocampus" / "merge_long.csv"
df_long = pd.read_csv(MERGE_LONG)


def nested_cv_one_col(df_long, *, col: str, seed: int, repeat_id: int) -> pd.DataFrame:
    task = TASKS["smci_pmci"]
    clinical = clinic.baseline_clinical_table(df_long)
    if clinical[col].isna().any():
        raise ValueError(f"NaN em {col}")
    pt = patient_labels_from_long(df_long, task)
    y = pt["y"].to_numpy(dtype=int)
    pts = pt["ID_PT"].astype(str).to_numpy()
    outer = StratifiedKFold(5, shuffle=True, random_state=seed)
    rows = []
    for fold, (tr_rel, te_rel) in enumerate(outer.split(np.zeros(len(y)), y), start=1):
        train_pts, test_pts = set(pts[tr_rel]), set(pts[te_rel])
        wide = patient_labels_from_long(df_long, task)[["ID_PT", "GROUP"]].merge(
            clinical, on="ID_PT", how="left",
        )
        wide["y"] = wide["GROUP"].map(task.label_map).astype(int)
        X = wide[[col]].astype(float)
        y_wide = wide["y"].to_numpy(dtype=int)
        tr = np.where(wide["ID_PT"].astype(str).isin(train_pts))[0]
        te = np.where(wide["ID_PT"].astype(str).isin(test_pts))[0]
        X_tr, X_te = X.iloc[tr], X.iloc[te]
        y_tr, y_te = y_wide[tr], y_wide[te]
        inner = StratifiedKFold(5, shuffle=True, random_state=seed + fold)
        tune_res = tune_pipeline(
            clinic._clinical_pipeline(MODEL_KEY, seed),
            X_tr, y_tr, inner,
            model_key=MODEL_KEY,
            selection_mode="raw",
            tuner="optuna",
            n_trials=OPTUNA_TRIALS,
            seed=seed + fold,
            n_jobs=gridsearch_n_jobs(MODEL_KEY),
            param_grid=clinic._clinical_param_grid(MODEL_KEY),
        )
        best = tune_res.estimator
        oof = cross_val_predict(best, X_tr, y_tr, cv=inner, method="predict_proba")[:, 1]
        thr = tune_youden_threshold(y_tr, oof)
        scores = best.predict_proba(X_te)[:, 1]
        preds = (scores >= thr).astype(int)
        selected = clinic._clinical_selected_names(
            MODEL_KEY, best.named_steps["clf"], [col],
        )
        rows.append({
            "feature_set": f"demo_{col.lower()}",
            "task": "smci_pmci",
            "modality": col.lower(),
            "model_key": MODEL_KEY,
            "with_combat": False,
            "selection_mode": "none",
            "repeat_id": repeat_id,
            "fold": fold,
            "test_id_pts": json.dumps(wide.iloc[te]["ID_PT"].astype(str).tolist()),
            "test_y_true": json.dumps(y_te.tolist()),
            "test_scores": json.dumps(scores.tolist()),
            "selected_features": json.dumps(selected),
            **fold_metrics(y_te, scores, preds),
        })
    return pd.DataFrame(rows)


def run_demo_col(col: str) -> pd.DataFrame:
    chunks = []
    for rid in repeat_ids(N_REPEATS):
        print(f"{col}  repeat {rid + 1}/{N_REPEATS}", flush=True)
        chunks.append(nested_cv_one_col(
            df_long, col=col,
            seed=cfg.seed + rid * 1000,
            repeat_id=rid,
        ))
    return prepare_ablation_df(pd.concat(chunks, ignore_index=True))


raw_age = run_demo_col("AGE")
raw_sex = run_demo_col("SEX")

pat_age = explode_patient_predictions(raw_age).groupby("ID_PT", as_index=False).agg(
    y=("y", "first"), score_age=("score", "mean"),
)
pat_sex = explode_patient_predictions(raw_sex).groupby("ID_PT", as_index=False).agg(
    y=("y", "first"), score_sex=("score", "mean"),
)
pt = pat_age.merge(pat_sex[["ID_PT", "score_sex"]], on="ID_PT")
y = pt["y"].to_numpy()
auc_age, p_age = permutation_auc_p(y, pt["score_age"].to_numpy(), n_perm=cfg.n_perm, seed=cfg.seed)
auc_sex, p_sex = permutation_auc_p(y, pt["score_sex"].to_numpy(), n_perm=cfg.n_perm, seed=cfg.seed + 1)
print(f"idade SVM  AUC={auc_age:.3f}  {ler_p(p_age)}")
print(f"sexo  SVM  AUC={auc_sex:.3f}  {ler_p(p_sex)}")

img = patient_image_scores(
    image_ablation_path(BASE, PROTOCOL_BASELINE, cfg.modality),
    cfg, expect_representation="t1_only",
).merge(pt, on=["ID_PT", "y"], how="inner")
y_m = img["y"].to_numpy()
auc_img, p_img = permutation_auc_p(y_m, img["score_img"].to_numpy(), n_perm=cfg.n_perm, seed=cfg.seed + 2)
d_age, lo_age, hi_age = bootstrap_auc_diff(
    y_m, img["score_img"].to_numpy(), img["score_age"].to_numpy(),
    n_boot=cfg.n_bootstrap, seed=cfg.seed + 3,
)
d_sex, lo_sex, hi_sex = bootstrap_auc_diff(
    y_m, img["score_img"].to_numpy(), img["score_sex"].to_numpy(),
    n_boot=cfg.n_bootstrap, seed=cfg.seed + 4,
)
confound = pd.DataFrame([
    {"modelo": "idade (SVM nested)", "auc": auc_age, "p_perm": p_age, "delta_vs_img": np.nan, "ci95_lo": np.nan, "ci95_hi": np.nan},
    {"modelo": "sexo (SVM nested)", "auc": auc_sex, "p_perm": p_sex, "delta_vs_img": np.nan, "ci95_lo": np.nan, "ci95_hi": np.nan},
    {"modelo": "shape T1", "auc": auc_img, "p_perm": p_img, "delta_vs_img": np.nan, "ci95_lo": np.nan, "ci95_hi": np.nan},
    {"modelo": "shape T1 − idade", "auc": d_age, "p_perm": np.nan, "delta_vs_img": d_age, "ci95_lo": lo_age, "ci95_hi": hi_age},
    {"modelo": "shape T1 − sexo", "auc": d_sex, "p_perm": np.nan, "delta_vs_img": d_sex, "ci95_lo": lo_sex, "ci95_hi": hi_sex},
])
save_table(confound, "stats_confound_48m6m")
display(confound.round(4))
print(f"shape T1 AUC={auc_img:.3f} n={len(img)}  {ler_p(p_img)}")
print(f"Δ vs idade {d_age:.3f} [{lo_age:.3f}, {hi_age:.3f}]")
print(f"Δ vs sexo  {d_sex:.3f} [{lo_sex:.3f}, {hi_sex:.3f}]")

AGE  repeat 1/10


/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=2000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=2000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=2000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=2000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/mnt/study-data/pgirardi/gra

AGE  repeat 2/10


/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=2000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=2000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=2000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=2000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/mnt/study-data/pgirardi/gra

AGE  repeat 3/10


...

AGE  repeat 9/10
AGE  repeat 10/10


/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=2000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=2000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=2000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=2000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/mnt/study-data/pgirardi/gra

SEX  repeat 1/10
SEX  repeat 2/10
SEX  repeat 3/10
SEX  repeat 4/10
SEX  repeat 5/10
SEX  repeat 6/10
SEX  repeat 7/10
SEX  repeat 8/10
SEX  repeat 9/10
SEX  repeat 10/10
idade SVM  AUC=0.395  não significativo (p=0.9910)
sexo  SVM  AUC=0.463  não significativo (p=0.8116)
Salvo: artigo/tables/stats_confound_48m6m.csv


,modelo,auc,p_perm,delta_vs_img,ci95_lo,ci95_hi
0,idade (SVM nested),0.3953,0.9910,NaN,NaN,NaN
1,sexo (SVM nested),0.4631,0.8116,NaN,NaN,NaN
2,shape T1,0.7115,0.0002,NaN,NaN,NaN
3,shape T1 − idade,0.3162,NaN,0.3162,0.2043,0.4292
4,shape T1 − sexo,0.2484,NaN,0.2484,0.1370,0.3626


shape T1 AUC=0.712 n=193  significativo (p=0.0002)
Δ vs idade 0.316 [0.204, 0.429]
Δ vs sexo  0.248 [0.137, 0.363]


## 3. Claim — Q4 vs T1 e D21 vs T1 em `48m_6m` (Tabela C)

Pareado por paciente. H1 one-sided: AUC(encoding) > AUC(T1).
FDR BH nas 5 famílias **dentro** de cada contraste.


In [14]:
cfg = STATS_CFG
print("Q4:", image_ablation_path(BASE, PROTOCOL_LONGITUDINAL, "vol").parent.parent.name)
print("D21:", image_ablation_path(BASE, PROTOCOL_TWO_VISIT, "vol").parent.parent.name)
print("T1:", image_ablation_path(BASE, PROTOCOL_BASELINE, "vol").parent.parent.name)
print("mods:", MODS_COMPARE)

cmp_claim = compare_q4_vs_t1(BASE, cfg, comparison=f"{COHORT_CLAIM}_t1_d21_d32_vs_t1_only")
save_table(cmp_claim, "stats_q4_vs_t1_48m6m")
print(f"\nClaim | {COHORT_CLAIM} | t1_d21_d32 vs t1_only | {cfg.task} | {cfg.model_key}\n")
display(cmp_claim.round(4))
print("\n── ΔAUC = Q4 − T1; FDR ──")
print_comparison_summary(cmp_claim, label_a="q4", label_b="t1_only")

cmp_claim_d21 = compare_d21_vs_t1(BASE, cfg, comparison=f"{COHORT_CLAIM}_t1_d21_vs_t1_only")
save_table(cmp_claim_d21, "stats_d21_vs_t1_48m6m")
print(f"\nClaim | {COHORT_CLAIM} | t1_d21 vs t1_only | {cfg.task} | {cfg.model_key}\n")
display(cmp_claim_d21.round(4))
print("\n── ΔAUC = D21 − T1; FDR ──")
print_comparison_summary(cmp_claim_d21, label_a="d21", label_b="t1_only")

for name, cmp, la in (
    ("Q4", cmp_claim, "q4"),
    ("D21", cmp_claim_d21, "d21"),
):
    if cmp.empty:
        continue
    print(f"\n── Mapa {name} ──")
    for _, r in cmp.iterrows():
        tag = "FDR+" if r["significant_fdr"] else ("raw+" if r["significant_raw"] else "n.s.")
        print(
            f"  {r['modality']}: {name}={r[f'auc_{la}']:.3f}  T1={r['auc_t1_only']:.3f}  "
            f"Δ={r['delta_auc']:+.3f}  [{r['ci95_lo']:.3f}, {r['ci95_hi']:.3f}]  "
            f"p={r['p_bootstrap_one_sided']:.4f}  q={r['p_fdr_bh']:.4f}  → {tag}"
        )


Q4: ablation_results_d21d32
D21: ablation_results_d21
T1: ablation_results_t1_only
mods: ('vol', 'shape', 'texture', 'disp', 'firstorder')
Salvo: artigo/tables/stats_q4_vs_t1_48m6m.csv

Claim | 48m_6m | t1_d21_d32 vs t1_only | smci_pmci | svm



,n_pacientes,auc_q4,auc_t1_only,delta_auc,ci95_lo,ci95_hi,p_bootstrap_one_sided,p_bootstrap_two_sided,q4_superior,p_perm_q4,p_perm_t1_only,significant_fdr,significant_raw,modality,comparison,p_fdr_bh
0,193,0.7630,0.7559,0.0071,-0.0351,0.0489,0.3639,0.7279,False,0.0002,0.0002,False,False,vol,48m_6m_t1_d21_d32_vs_t1_only,0.6065
1,193,0.7213,0.7115,0.0098,-0.0300,0.0487,0.2941,0.5883,False,0.0002,0.0002,False,False,shape,48m_6m_t1_d21_d32_vs_t1_only,0.6065
2,193,0.6742,0.6276,0.0466,-0.0221,0.1154,0.0946,0.1892,False,0.0002,0.0016,False,False,texture,48m_6m_t1_d21_d32_vs_t1_only,0.4729
3,193,0.5771,0.5952,-0.0182,-0.0642,0.0261,0.7904,0.4191,False,0.0374,0.0124,False,False,disp,48m_6m_t1_d21_d32_vs_t1_only,0.7904
4,193,0.6861,0.6865,-0.0005,-0.0287,0.0299,0.5261,0.9478,False,0.0002,0.0002,False,False,firstorder,48m_6m_t1_d21_d32_vs_t1_only,0.6576



── ΔAUC = Q4 − T1; FDR ──
  vol: ΔAUC=0.007 [-0.035, 0.049]  p=0.3639  q=0.6065  → sem evidência
  shape: ΔAUC=0.010 [-0.030, 0.049]  p=0.2941  q=0.6065  → sem evidência
  texture: ΔAUC=0.047 [-0.022, 0.115]  p=0.0946  q=0.4729  → sem evidência
  disp: ΔAUC=-0.018 [-0.064, 0.026]  p=0.7904  q=0.7904  → sem evidência
  firstorder: ΔAUC=-0.000 [-0.029, 0.030]  p=0.5261  q=0.6576  → sem evidência
Salvo: artigo/tables/stats_d21_vs_t1_48m6m.csv

Claim | 48m_6m | t1_d21 vs t1_only | smci_pmci | svm



,n_pacientes,auc_d21,auc_t1_only,delta_auc,ci95_lo,ci95_hi,p_bootstrap_one_sided,p_bootstrap_two_sided,d21_superior,p_perm_d21,p_perm_t1_only,significant_fdr,significant_raw,modality,comparison,p_fdr_bh
0,193,0.7400,0.7559,-0.0160,-0.0373,0.0037,0.9396,0.1208,False,0.0002,0.0002,False,False,vol,48m_6m_t1_d21_vs_t1_only,0.9986
1,193,0.7160,0.7115,0.0045,-0.0210,0.0306,0.3693,0.7387,False,0.0002,0.0002,False,False,shape,48m_6m_t1_d21_vs_t1_only,0.6155
2,193,0.6705,0.6276,0.0429,-0.0238,0.1082,0.1042,0.2084,False,0.0002,0.0016,False,False,texture,48m_6m_t1_d21_vs_t1_only,0.5209
3,193,0.5478,0.5952,-0.0474,-0.0815,-0.0148,0.9986,0.0028,False,0.1344,0.0124,False,False,disp,48m_6m_t1_d21_vs_t1_only,0.9986
4,193,0.6905,0.6865,0.0040,-0.0162,0.0248,0.3639,0.7279,False,0.0002,0.0002,False,False,firstorder,48m_6m_t1_d21_vs_t1_only,0.6155



── ΔAUC = D21 − T1; FDR ──
  vol: ΔAUC=-0.016 [-0.037, 0.004]  p=0.9396  q=0.9986  → sem evidência
  shape: ΔAUC=0.004 [-0.021, 0.031]  p=0.3693  q=0.6155  → sem evidência
  texture: ΔAUC=0.043 [-0.024, 0.108]  p=0.1042  q=0.5209  → sem evidência
  disp: ΔAUC=-0.047 [-0.081, -0.015]  p=0.9986  q=0.9986  → sem evidência
  firstorder: ΔAUC=0.004 [-0.016, 0.025]  p=0.3639  q=0.6155  → sem evidência

── Mapa Q4 ──
  vol: Q4=0.763  T1=0.756  Δ=+0.007  [-0.035, 0.049]  p=0.3639  q=0.6065  → n.s.
  shape: Q4=0.721  T1=0.712  Δ=+0.010  [-0.030, 0.049]  p=0.2941  q=0.6065  → n.s.
  texture: Q4=0.674  T1=0.628  Δ=+0.047  [-0.022, 0.115]  p=0.0946  q=0.4729  → n.s.
  disp: Q4=0.577  T1=0.595  Δ=-0.018  [-0.064, 0.026]  p=0.7904  q=0.7904  → n.s.
  firstorder: Q4=0.686  T1=0.687  Δ=-0.000  [-0.029, 0.030]  p=0.5261  q=0.6576  → n.s.

── Mapa D21 ──
  vol: D21=0.740  T1=0.756  Δ=-0.016  [-0.037, 0.004]  p=0.9396  q=0.9986  → n.s.
  shape: D21=0.716  T1=0.712  Δ=+0.004  [-0.021, 0.031]  p=0.3693  q

## 4. Sensibilidade — Q4 vs T1 e D21 vs T1 × 4 coortes

Células sobrepõem sujeitos. `36m_12m` inverte (pMCI n=33). Não é réplica.
FDR **dentro** de cada coorte × contraste (5 famílias).


In [15]:
cfg = STATS_CFG
cols = [
    "cohort", "modality", "n_pacientes",
    "auc_long", "auc_t1_only", "delta_auc",
    "ci95_lo", "ci95_hi", "p_bootstrap_one_sided", "p_fdr_bh",
    "significant_raw", "significant_fdr", "contrast",
]


def _gradient(contrast: str, runner, label_a: str, cache_claim):
    rows = []
    for cname in COHORTS_GRADIENT:
        base = Path(f"csvs/cohorts/{cname}")
        if cname == COHORT_CLAIM and cache_claim is not None and not cache_claim.empty:
            cmp = cache_claim.copy()
        else:
            cmp = runner(base, cfg, comparison=f"{cname}_{contrast}")
        if cmp.empty:
            print(f"{cname} ({contrast}): sem dados")
            continue
        cmp = cmp.copy()
        cmp["cohort"] = cname
        cmp["contrast"] = contrast
        cmp["auc_long"] = cmp[f"auc_{label_a}"]
        rows.append(cmp)
        print(f"\n=== {cname} | {contrast} ===")
        print_comparison_summary(cmp, label_a=label_a, label_b="t1_only")
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


cmp_gradient = _gradient(
    "t1_d21_d32_vs_t1_only", compare_q4_vs_t1, "q4",
    cmp_claim if "cmp_claim" in dir() else None,
)
if not cmp_gradient.empty:
    save_table(cmp_gradient[cols], "stats_q4_vs_t1_gradient")
    display(cmp_gradient[cols].round(4))
    print("\n── pivot ΔAUC Q4−T1 ──")
    display(
        cmp_gradient.pivot(index="modality", columns="cohort", values="delta_auc")
        .reindex(index=list(MODS_COMPARE), columns=list(COHORTS_GRADIENT))
        .round(3)
    )

cmp_gradient_d21 = _gradient(
    "t1_d21_vs_t1_only", compare_d21_vs_t1, "d21",
    cmp_claim_d21 if "cmp_claim_d21" in dir() else None,
)
if not cmp_gradient_d21.empty:
    save_table(cmp_gradient_d21[cols], "stats_d21_vs_t1_gradient")
    display(cmp_gradient_d21[cols].round(4))
    print("\n── pivot ΔAUC D21−T1 ──")
    display(
        cmp_gradient_d21.pivot(index="modality", columns="cohort", values="delta_auc")
        .reindex(index=list(MODS_COMPARE), columns=list(COHORTS_GRADIENT))
        .round(3)
    )



=== 36m_6m | t1_d21_d32_vs_t1_only ===
  vol: ΔAUC=0.005 [-0.025, 0.034]  p=0.3803  q=0.4754  → sem evidência
  shape: ΔAUC=0.010 [-0.022, 0.042]  p=0.2655  q=0.4426  → sem evidência
  texture: ΔAUC=0.070 [0.005, 0.136]  p=0.0178  q=0.0890  → raw sig.
  disp: ΔAUC=-0.029 [-0.077, 0.019]  p=0.8826  q=0.8826  → sem evidência
  firstorder: ΔAUC=0.016 [-0.013, 0.045]  p=0.1468  q=0.3669  → sem evidência

=== 36m_12m | t1_d21_d32_vs_t1_only ===
  vol: ΔAUC=-0.006 [-0.087, 0.074]  p=0.5521  q=0.6901  → sem evidência
  shape: ΔAUC=0.045 [-0.012, 0.104]  p=0.0582  q=0.1455  → sem evidência
  texture: ΔAUC=0.016 [-0.074, 0.110]  p=0.3591  q=0.5985  → sem evidência
  disp: ΔAUC=0.178 [0.053, 0.300]  p=0.0042  q=0.0210  → FDR sig.
  firstorder: ΔAUC=-0.025 [-0.119, 0.072]  p=0.6953  q=0.6953  → sem evidência

=== 48m_6m | t1_d21_d32_vs_t1_only ===
  vol: ΔAUC=0.007 [-0.035, 0.049]  p=0.3639  q=0.6065  → sem evidência
  shape: ΔAUC=0.010 [-0.030, 0.049]  p=0.2941  q=0.6065  → sem evidência
  text

,cohort,modality,n_pacientes,auc_long,auc_t1_only,delta_auc,ci95_lo,ci95_hi,p_bootstrap_one_sided,p_fdr_bh,significant_raw,significant_fdr,contrast
0,36m_6m,vol,231,0.7703,0.7656,0.0047,-0.0250,0.0336,0.3803,0.4754,False,False,t1_d21_d32_vs_t1_only
1,36m_6m,shape,231,0.7433,0.7329,0.0104,-0.0217,0.0424,0.2655,0.4426,False,False,t1_d21_d32_vs_t1_only
2,36m_6m,texture,231,0.6891,0.6186,0.0705,0.0046,0.1362,0.0178,0.0890,True,False,t1_d21_d32_vs_t1_only
3,36m_6m,disp,231,0.5737,0.6029,-0.0293,-0.0768,0.0192,0.8826,0.8826,False,False,t1_d21_d32_vs_t1_only
4,36m_6m,firstorder,231,0.6695,0.6538,0.0157,-0.0134,0.0455,0.1468,0.3669,False,False,t1_d21_d32_vs_t1_only
5,36m_12m,vol,154,0.6837,0.6900,-0.0063,-0.0870,0.0744,0.5521,0.6901,False,False,t1_d21_d32_vs_t1_only
6,36m_12m,shape,154,0.7250,0.6804,0.0446,-0.0121,0.1040,0.0582,0.1455,False,False,t1_d21_d32_vs_t1_only
7,36m_12m,texture,154,0.5159,0.4996,0.0163,-0.0743,0.1095,0.3591,0.5985,False,False,t1_d21_d32_vs_t1_only
8,36m_12m,disp,154,0.5167,0.3386,0.1781,0.0528,0.2997,0.0042,0.0210,True,True,t1_d21_d32_vs_t1_only
9,36m_12m,firstorder,154,0.5733,0.5983,-0.0250,-0.1187,0.0723,0.6953,0.6953,False,False,t1_d21_d32_vs_t1_only



── pivot ΔAUC Q4−T1 ──


cohort,36m_6m,36m_12m,48m_6m,48m_12m
modality,,,,
vol,0.005,-0.006,0.007,0.089
shape,0.010,0.045,0.010,0.002
texture,0.070,0.016,0.047,0.048
disp,-0.029,0.178,-0.018,-0.016
firstorder,0.016,-0.025,-0.000,0.017



=== 36m_6m | t1_d21_vs_t1_only ===
  vol: ΔAUC=-0.014 [-0.029, -0.000]  p=0.9776  q=0.9994  → sem evidência
  shape: ΔAUC=0.011 [-0.016, 0.040]  p=0.2156  q=0.5389  → sem evidência
  texture: ΔAUC=0.072 [0.007, 0.139]  p=0.0136  q=0.0680  → raw sig.
  disp: ΔAUC=-0.050 [-0.080, -0.020]  p=0.9994  q=0.9994  → sem evidência
  firstorder: ΔAUC=0.002 [-0.018, 0.023]  p=0.4375  q=0.7292  → sem evidência

=== 36m_12m | t1_d21_vs_t1_only ===
  vol: ΔAUC=0.030 [-0.040, 0.103]  p=0.2022  q=0.3593  → sem evidência
  shape: ΔAUC=0.012 [-0.036, 0.063]  p=0.3267  q=0.4084  → sem evidência
  texture: ΔAUC=0.025 [-0.042, 0.091]  p=0.2156  q=0.3593  → sem evidência
  disp: ΔAUC=0.161 [0.016, 0.300]  p=0.0146  q=0.0730  → raw sig.
  firstorder: ΔAUC=-0.061 [-0.117, -0.003]  p=0.9792  q=0.9792  → sem evidência

=== 48m_6m | t1_d21_vs_t1_only ===
  vol: ΔAUC=-0.016 [-0.037, 0.004]  p=0.9396  q=0.9986  → sem evidência
  shape: ΔAUC=0.004 [-0.021, 0.031]  p=0.3693  q=0.6155  → sem evidência
  texture: ΔAU

,cohort,modality,n_pacientes,auc_long,auc_t1_only,delta_auc,ci95_lo,ci95_hi,p_bootstrap_one_sided,p_fdr_bh,significant_raw,significant_fdr,contrast
0,36m_6m,vol,231,0.7513,0.7656,-0.0143,-0.0291,-0.0004,0.9776,0.9994,False,False,t1_d21_vs_t1_only
1,36m_6m,shape,231,0.7436,0.7329,0.0107,-0.0158,0.0397,0.2156,0.5389,False,False,t1_d21_vs_t1_only
2,36m_6m,texture,231,0.6911,0.6186,0.0725,0.0072,0.1392,0.0136,0.0680,True,False,t1_d21_vs_t1_only
3,36m_6m,disp,231,0.5530,0.6029,-0.0500,-0.0802,-0.0201,0.9994,0.9994,False,False,t1_d21_vs_t1_only
4,36m_6m,firstorder,231,0.6557,0.6538,0.0019,-0.0176,0.0231,0.4375,0.7292,False,False,t1_d21_vs_t1_only
5,36m_12m,vol,154,0.7195,0.6900,0.0296,-0.0397,0.1035,0.2022,0.3593,False,False,t1_d21_vs_t1_only
6,36m_12m,shape,154,0.6920,0.6804,0.0115,-0.0359,0.0629,0.3267,0.4084,False,False,t1_d21_vs_t1_only
7,36m_12m,texture,154,0.5247,0.4996,0.0250,-0.0417,0.0909,0.2156,0.3593,False,False,t1_d21_vs_t1_only
8,36m_12m,disp,154,0.4991,0.3386,0.1605,0.0155,0.2997,0.0146,0.0730,True,False,t1_d21_vs_t1_only
9,36m_12m,firstorder,154,0.5369,0.5983,-0.0614,-0.1172,-0.0028,0.9792,0.9792,False,False,t1_d21_vs_t1_only



── pivot ΔAUC D21−T1 ──


cohort,36m_6m,36m_12m,48m_6m,48m_12m
modality,,,,
vol,-0.014,0.030,-0.016,0.037
shape,0.011,0.012,0.004,-0.004
texture,0.072,0.025,0.043,-0.109
disp,-0.050,0.161,-0.047,-0.005
firstorder,0.002,-0.061,0.004,-0.000


## 5. Late vs teto unimodal (shape T1)

Specs: união T1, união Q4, âncora (`shape T1 ∪ resto Q4`). ΔAUC = late − shape T1.


In [16]:
cfg = STATS_CFG
shape_cfg = cfg_for_modality("shape", cfg)
shape_path = image_ablation_path(BASE, PROTOCOL_BASELINE, "shape")
pat_shape = patient_scores_from_path(shape_path, shape_cfg)

late_rows = []
for i, (lab, proto) in enumerate((
    ("late T1", LATE_ALL_T1),
    ("late Q4", LATE_ALL_Q4),
    ("late ancora", LATE_ANCORA),
)):
    path = image_ablation_path(BASE, proto, proto)
    if not path.is_file():
        print("MISSING", path)
        continue
    late_cfg = cfg_for_modality(proto, cfg, protocol=proto)
    pat_late = patient_scores_from_path(path, late_cfg)
    paired = pat_late.merge(pat_shape, on=["ID_PT", "y"], suffixes=("_late", "_shape"))
    if paired.empty:
        print("sem pares", lab)
        continue
    row = paired_comparison_row(
        paired,
        score_a="score_late",
        score_b="score_shape",
        label_a="late",
        label_b="shape_t1",
        n_boot=cfg.n_bootstrap,
        seed=cfg.seed + 400 + i,
        permutation_auc_p=permutation_auc_p,
        n_perm=cfg.n_perm,
        alpha=ALPHA,
    )
    row.update({"spec": lab, "protocol": proto, "modality": lab})
    late_rows.append(row)

cmp_late = pd.DataFrame(late_rows)
if not cmp_late.empty:
    cmp_late["p_fdr_bh"] = apply_bh_fdr(cmp_late["p_bootstrap_one_sided"].to_numpy())
    cmp_late["significant_fdr"] = (cmp_late["p_fdr_bh"] < ALPHA) & (cmp_late["ci95_lo"] > 0)
    save_table(cmp_late, "stats_late_vs_shape_48m6m")
    display(cmp_late.round(4))
    print("\n── ΔAUC = late − shape T1 ──")
    print_comparison_summary(cmp_late, label_a="late", label_b="shape_t1")


Salvo: artigo/tables/stats_late_vs_shape_48m6m.csv


,n_pacientes,auc_late,auc_shape_t1,delta_auc,ci95_lo,ci95_hi,p_bootstrap_one_sided,p_bootstrap_two_sided,late_superior,p_perm_late,p_perm_shape_t1,significant_fdr,significant_raw,spec,protocol,modality,p_fdr_bh
0,193,0.7718,0.7115,0.0603,0.0164,0.1060,0.0036,0.0072,True,0.0002,0.0002,True,True,late T1,late__t1_vol__t1_shape__t1_texture__t1_disp__t...,late T1,0.0038
1,193,0.7872,0.7115,0.0757,0.0205,0.1304,0.0038,0.0076,True,0.0002,0.0002,True,True,late Q4,late__t1_d21d32_vol__t1_d21d32_shape__t1_d21d3...,late Q4,0.0038
2,193,0.8232,0.7115,0.1116,0.0569,0.1665,0.0002,0.0004,True,0.0002,0.0002,True,True,late ancora,late__t1_shape__t1_d21d32_vol__t1_d21d32_textu...,late ancora,0.0006



── ΔAUC = late − shape T1 ──
  late T1: ΔAUC=0.060 [0.016, 0.106]  p=0.0036  q=0.0038  → FDR sig.
  late Q4: ΔAUC=0.076 [0.020, 0.130]  p=0.0038  q=0.0038  → FDR sig.
  late ancora: ΔAUC=0.112 [0.057, 0.166]  p=0.0002  q=0.0006  → FDR sig.


## 6. Clínico vs vol T1 (`48m_6m`)

Imagem = **vol** `t1_only`. Fusão = clinic + vol T1.
Δ chave = (clinic+img) − clinic (IC bootstrap).


In [17]:
cfg = STATS_CFG
mod = CLINIC_MODALITY
img_cfg = cfg_for_modality(mod, cfg)
clin_cfg = cfg_clinical(cfg)
fusion_path = fusion_results_path(
    BASE, mod, selection_mode=cfg.selection_mode, with_combat=cfg.with_combat,
    representation="t1_only",
)
clin_path = clinical_results_path(BASE)
img_path = image_ablation_path(BASE, PROTOCOL_BASELINE, mod)

clinical_rows = []
pairs = [
    (img_path, clin_path, img_cfg, clin_cfg, "score_img", "score_clin", "img", "clin", "vol_t1_vs_clinical"),
    (fusion_path, clin_path, img_cfg, clin_cfg, "score_fusion", "score_clin", "fusion", "clin", "fusion_vs_clinical"),
    (fusion_path, img_path, img_cfg, img_cfg, "score_fusion", "score_img", "fusion", "img", "fusion_vs_vol_t1"),
]
for i, (pa, pb, ca, cb, sa, sb, la, lb, tag) in enumerate(pairs):
    if not pa.exists() or not pb.exists():
        print("MISSING", pa if not pa.exists() else pb)
        continue
    paired = patient_scores_from_path(pa, ca).merge(
        patient_scores_from_path(pb, cb), on=["ID_PT", "y"], suffixes=(f"_{la}", f"_{lb}"),
    )
    # suffixes only apply when column names collide; scores both named "score"
    if f"score_{la}" not in paired.columns:
        paired = patient_scores_from_path(pa, ca).rename(columns={"score": sa}).merge(
            patient_scores_from_path(pb, cb).rename(columns={"score": sb}),
            on=["ID_PT", "y"],
        )
    row = paired_comparison_row(
        paired, score_a=sa, score_b=sb, label_a=la, label_b=lb,
        n_boot=cfg.n_bootstrap, seed=cfg.seed + 200 + i,
        permutation_auc_p=permutation_auc_p, n_perm=cfg.n_perm, alpha=ALPHA,
    )
    row.update({"modality": mod, "comparison": tag})
    clinical_rows.append(row)

cmp_clinical = pd.DataFrame(clinical_rows)
if not cmp_clinical.empty:
    cmp_clinical["p_fdr_bh"] = apply_bh_fdr(cmp_clinical["p_bootstrap_one_sided"].to_numpy())
    cmp_clinical["significant_fdr"] = (
        (cmp_clinical["p_fdr_bh"] < ALPHA) & (cmp_clinical["ci95_lo"] > 0)
    )
    save_table(cmp_clinical, "stats_clinic_48m6m")
    display(cmp_clinical.round(4))
    for _, r in cmp_clinical.iterrows():
        print(
            f"  {r['comparison']}: Δ={r['delta_auc']:.3f} "
            f"[{r['ci95_lo']:.3f}, {r['ci95_hi']:.3f}]  p={r['p_bootstrap_one_sided']:.4f}"
        )


Salvo: artigo/tables/stats_clinic_48m6m.csv


,n_pacientes,auc_img,auc_clin,delta_auc,ci95_lo,ci95_hi,p_bootstrap_one_sided,p_bootstrap_two_sided,img_superior,p_perm_img,p_perm_clin,significant_fdr,significant_raw,modality,comparison,auc_fusion,fusion_superior,p_perm_fusion,p_fdr_bh
0,193,0.7559,0.8396,-0.0837,-0.1650,-0.0035,0.9814,0.0372,False,0.0002,0.0002,False,False,vol,vol_t1_vs_clinical,NaN,NaN,NaN,0.9814
1,193,NaN,0.8396,0.0263,-0.0124,0.0640,0.0896,0.1792,NaN,NaN,0.0002,False,False,vol,fusion_vs_clinical,0.8659,False,0.0002,0.1344
2,193,0.7559,NaN,0.1099,0.0548,0.1652,0.0002,0.0004,NaN,0.0002,NaN,True,True,vol,fusion_vs_vol_t1,0.8659,True,0.0002,0.0006


  vol_t1_vs_clinical: Δ=-0.084 [-0.165, -0.004]  p=0.9814
  fusion_vs_clinical: Δ=0.026 [-0.012, 0.064]  p=0.0896
  fusion_vs_vol_t1: Δ=0.110 [0.055, 0.165]  p=0.0002


## 7. Leaky — vol Q4 vs vol Q4 com stats globais

`t1_d21_d32` vs `t1_d21_d32_global` (só vol no disco). ΔAUC = Q4 correcto − leaky.


In [18]:
cfg = STATS_CFG
cmp_leaky = compare_modalities(
    ("vol",),
    path_a=lambda m: image_ablation_path(BASE, PROTOCOL_LONGITUDINAL, m),
    path_b=lambda m: image_ablation_path(BASE, "t1_d21_d32_global", m),
    cfg_for_mod=lambda m: cfg_for_modality(m, cfg),
    load_patients=patient_scores_from_path,
    permutation_auc_p=permutation_auc_p,
    n_perm=cfg.n_perm,
    n_bootstrap=cfg.n_bootstrap,
    seed=cfg.seed + 100,
    label_a="q4",
    label_b="leaky",
    comparison="q4_vs_leaky_vol",
    alpha=ALPHA,
)
save_table(cmp_leaky, "stats_leaky_48m6m")
display(cmp_leaky.round(4))
print("── ΔAUC = Q4 − leaky (negativo → leaky maior) ──")
print_comparison_summary(cmp_leaky, label_a="q4", label_b="leaky")


Salvo: artigo/tables/stats_leaky_48m6m.csv


,n_pacientes,auc_q4,auc_leaky,delta_auc,ci95_lo,ci95_hi,p_bootstrap_one_sided,p_bootstrap_two_sided,q4_superior,p_perm_q4,p_perm_leaky,significant_fdr,significant_raw,modality,comparison,p_fdr_bh
0,193,0.763,0.7702,-0.0072,-0.0192,0.0046,0.882,0.236,False,0.0002,0.0002,False,False,vol,q4_vs_leaky_vol,0.882


── ΔAUC = Q4 − leaky (negativo → leaky maior) ──
  vol: ΔAUC=-0.007 [-0.019, 0.005]  p=0.8820  q=0.8820  → sem evidência


## 8. Soft True vs False (descritivo)

Pastas: `48m_6m` (120 pMCI) vs `48m_6m_soft_False` (74 pMCI).
SVM · T1 e Q4 · 5 famílias. **Sem** bootstrap pareado (n ≠; pacientes diferentes).
Δ = AUC(soft True) − AUC(soft False).


In [19]:
cfg = STATS_CFG
base_soft = Path(f"csvs/cohorts/{SOFT_FALSE_COHORT}")
rows_soft = []
for proto, lab in (
    (PROTOCOL_BASELINE, "t1_only"),
    (PROTOCOL_LONGITUDINAL, "t1_d21_d32"),
):
    for mod in MODS_COMPARE:
        cfg_m = cfg_for_modality(mod, cfg)
        p_true = image_ablation_path(BASE, proto, mod)
        p_false = image_ablation_path(base_soft, proto, mod)
        if not p_true.is_file() or not p_false.is_file():
            print("MISSING", proto, mod, p_true.is_file(), p_false.is_file())
            continue
        auc_t, n_t = patient_auc(p_true, cfg_m)
        auc_f, n_f = patient_auc(p_false, cfg_m)
        rows_soft.append({
            "protocol": lab,
            "modality": mod,
            "auc_soft_true": auc_t,
            "n_soft_true": n_t,
            "auc_soft_false": auc_f,
            "n_soft_false": n_f,
            "delta_true_minus_false": auc_t - auc_f,
        })

cmp_soft = pd.DataFrame(rows_soft)
if not cmp_soft.empty:
    save_table(cmp_soft, "stats_soft_true_vs_false")
    display(cmp_soft.round(4))
    print("\n── pivot Δ (True−False) ──")
    display(
        cmp_soft.pivot(index="modality", columns="protocol", values="delta_true_minus_false")
        .reindex(index=list(MODS_COMPARE))
        .round(3)
    )


Salvo: artigo/tables/stats_soft_true_vs_false.csv


,protocol,modality,auc_soft_true,n_soft_true,auc_soft_false,n_soft_false,delta_true_minus_false
0,t1_only,vol,0.7559,193,0.7381,147,0.0179
1,t1_only,shape,0.7115,193,0.7066,147,0.0049
2,t1_only,texture,0.6276,193,0.6261,147,0.0016
3,t1_only,disp,0.5952,193,0.5017,147,0.0935
4,t1_only,firstorder,0.6865,193,0.6973,147,-0.0108
5,t1_d21_d32,vol,0.7630,193,0.7282,147,0.0348
6,t1_d21_d32,shape,0.7213,193,0.6864,147,0.0349
7,t1_d21_d32,texture,0.6742,193,0.6040,147,0.0702
8,t1_d21_d32,disp,0.5771,193,0.6237,147,-0.0466
9,t1_d21_d32,firstorder,0.6861,193,0.6707,147,0.0154



── pivot Δ (True−False) ──


protocol,t1_d21_d32,t1_only
modality,,
vol,0.035,0.018
shape,0.035,0.005
texture,0.070,0.002
disp,-0.047,0.094
firstorder,0.015,-0.011


## 9. ComBat vs nocombat (sensibilidade, claim)

Pastas `ablation_results_combat_*` (with_combat=True) vs pastas padrão nocombat.
Pareado por paciente na **mesma** coorte `48m_6m`. T1 e Q4 · 5 famílias · SVM.


In [20]:
cfg = STATS_CFG
rows_c = []
for i, (proto, lab) in enumerate((
    (PROTOCOL_BASELINE, "t1_only"),
    (PROTOCOL_LONGITUDINAL, "t1_d21_d32"),
)):
    for j, mod in enumerate(MODS_COMPARE):
        p_c = combat_ablation_path(BASE, proto, mod)
        p_n = image_ablation_path(BASE, proto, mod)
        if not p_c.is_file() or not p_n.is_file():
            print("MISSING combat", proto, mod)
            continue
        cfg_c = cfg_for_modality(mod, cfg, with_combat=True)
        cfg_n = cfg_for_modality(mod, cfg, with_combat=False)
        paired = patient_scores_from_path(p_c, cfg_c).rename(columns={"score": "score_combat"}).merge(
            patient_scores_from_path(p_n, cfg_n).rename(columns={"score": "score_nocombat"}),
            on=["ID_PT", "y"],
        )
        if paired.empty:
            print("sem pares", proto, mod)
            continue
        row = paired_comparison_row(
            paired,
            score_a="score_combat",
            score_b="score_nocombat",
            label_a="combat",
            label_b="nocombat",
            n_boot=cfg.n_bootstrap,
            seed=cfg.seed + 700 + 10 * i + j,
            permutation_auc_p=permutation_auc_p,
            n_perm=cfg.n_perm,
            alpha=ALPHA,
        )
        row.update({"protocol": lab, "modality": mod})
        rows_c.append(row)

cmp_combat = pd.DataFrame(rows_c)
if not cmp_combat.empty:
    # FDR dentro de cada protocol (5 fam)
    cmp_combat["p_fdr_bh"] = np.nan
    for lab, idx in cmp_combat.groupby("protocol").groups.items():
        q = apply_bh_fdr(cmp_combat.loc[idx, "p_bootstrap_one_sided"].to_numpy())
        cmp_combat.loc[idx, "p_fdr_bh"] = q
    cmp_combat["significant_fdr"] = (cmp_combat["p_fdr_bh"] < ALPHA) & (cmp_combat["ci95_lo"] > 0)
    save_table(cmp_combat, "stats_combat_vs_nocombat")
    display(cmp_combat.round(4))
    print("── ΔAUC = combat − nocombat ──")
    print_comparison_summary(cmp_combat, label_a="combat", label_b="nocombat")


Salvo: artigo/tables/stats_combat_vs_nocombat.csv


,n_pacientes,auc_combat,auc_nocombat,delta_auc,ci95_lo,ci95_hi,p_bootstrap_one_sided,p_bootstrap_two_sided,combat_superior,p_perm_combat,p_perm_nocombat,significant_fdr,significant_raw,protocol,modality,p_fdr_bh
0,193,0.7376,0.7559,-0.0184,-0.0540,0.0173,0.8474,0.3051,False,0.0002,0.0002,False,False,t1_only,vol,0.9922
1,193,0.7261,0.7115,0.0146,-0.0123,0.0427,0.1484,0.2967,False,0.0002,0.0002,False,False,t1_only,shape,0.3709
2,193,0.6672,0.6276,0.0396,-0.0111,0.0896,0.0590,0.1180,False,0.0004,0.0012,False,False,t1_only,texture,0.2949
3,193,0.5291,0.5952,-0.0661,-0.1204,-0.0135,0.9922,0.0156,False,0.2438,0.0144,False,False,t1_only,disp,0.9922
4,193,0.6705,0.6865,-0.0160,-0.0776,0.0436,0.7017,0.5967,False,0.0002,0.0002,False,False,t1_only,firstorder,0.9922
5,193,0.7358,0.7630,-0.0272,-0.0585,0.0041,0.9548,0.0904,False,0.0002,0.0002,False,False,t1_d21_d32,vol,0.9548
6,193,0.7396,0.7213,0.0183,-0.0075,0.0449,0.0860,0.1720,False,0.0002,0.0002,False,False,t1_d21_d32,shape,0.2285
7,193,0.7050,0.6742,0.0308,-0.0135,0.0768,0.0914,0.1828,False,0.0002,0.0002,False,False,t1_d21_d32,texture,0.2285
8,193,0.5655,0.5771,-0.0115,-0.0498,0.0269,0.7275,0.5451,False,0.0652,0.0372,False,False,t1_d21_d32,disp,0.9548
9,193,0.6555,0.6861,-0.0306,-0.0837,0.0237,0.8658,0.2683,False,0.0006,0.0002,False,False,t1_d21_d32,firstorder,0.9548


── ΔAUC = combat − nocombat ──
  vol: ΔAUC=-0.018 [-0.054, 0.017]  p=0.8474  q=0.9922  → sem evidência
  shape: ΔAUC=0.015 [-0.012, 0.043]  p=0.1484  q=0.3709  → sem evidência
  texture: ΔAUC=0.040 [-0.011, 0.090]  p=0.0590  q=0.2949  → sem evidência
  disp: ΔAUC=-0.066 [-0.120, -0.014]  p=0.9922  q=0.9922  → sem evidência
  firstorder: ΔAUC=-0.016 [-0.078, 0.044]  p=0.7017  q=0.9922  → sem evidência
  vol: ΔAUC=-0.027 [-0.058, 0.004]  p=0.9548  q=0.9548  → sem evidência
  shape: ΔAUC=0.018 [-0.008, 0.045]  p=0.0860  q=0.2285  → sem evidência
  texture: ΔAUC=0.031 [-0.013, 0.077]  p=0.0914  q=0.2285  → sem evidência
  disp: ΔAUC=-0.012 [-0.050, 0.027]  p=0.7275  q=0.9548  → sem evidência
  firstorder: ΔAUC=-0.031 [-0.084, 0.024]  p=0.8658  q=0.9548  → sem evidência


## 10. Tabela D — resumo complementares (claim)

Uma linha por análise: late âncora, leaky, clinic, clinic+vol, ComBat, soft.


In [21]:
rows_d = []

def _add(analysis, detail, auc=None, delta=None, note=""):
    rows_d.append({
        "analysis": analysis,
        "detail": detail,
        "auc": auc,
        "delta": delta,
        "note": note,
    })

# late ancora
if "cmp_late" in dir() and not cmp_late.empty:
    r = cmp_late.loc[cmp_late["spec"] == "late ancora"].iloc[0]
    _add("late_ancora", LATE_ANCORA, auc=r.get("auc_late"), delta=r.get("delta_auc"),
         note="Δ vs shape T1")

# leaky
if "cmp_leaky" in dir() and not cmp_leaky.empty:
    r = cmp_leaky.iloc[0]
    _add("leaky_vol_q4", "vol t1_d21_d32_global", auc=r.get("auc_leaky"),
         delta=r.get("delta_auc"), note="Δ = Q4 − leaky (neg → leaky maior)")

# clinic
if "cmp_clinical" in dir() and not cmp_clinical.empty:
    for tag in ("vol_t1_vs_clinical", "fusion_vs_clinical", "fusion_vs_vol_t1"):
        sub = cmp_clinical.loc[cmp_clinical["comparison"] == tag]
        if sub.empty:
            continue
        r = sub.iloc[0]
        _add("clinic", tag, auc=None, delta=r.get("delta_auc"),
             note=f"[{r['ci95_lo']:.3f},{r['ci95_hi']:.3f}]")

# combat mean |Δ| by protocol (descriptive row)
if "cmp_combat" in dir() and not cmp_combat.empty:
    for lab, g in cmp_combat.groupby("protocol"):
        _add("combat_vs_nocombat", lab,
             auc=None,
             delta=float(g["delta_auc"].mean()),
             note=f"mean ΔAUC over 5 fam; n_sig_fdr={int(g['significant_fdr'].sum())}")

# soft mean Δ
if "cmp_soft" in dir() and not cmp_soft.empty:
    for lab, g in cmp_soft.groupby("protocol"):
        _add("soft_true_vs_false", lab,
             auc=None,
             delta=float(g["delta_true_minus_false"].mean()),
             note="descriptive mean Δ over 5 fam; n_pMCI 120 vs 74")

table_d = pd.DataFrame(rows_d)
save_table(table_d, "stats_table_d_complements")
display(table_d.round(4))


Salvo: artigo/tables/stats_table_d_complements.csv


,analysis,detail,auc,delta,note
0,late_ancora,late__t1_shape__t1_d21d32_vol__t1_d21d32_textu...,0.8232,0.1116,Δ vs shape T1
1,leaky_vol_q4,vol t1_d21_d32_global,0.7702,-0.0072,Δ = Q4 − leaky (neg → leaky maior)
2,clinic,vol_t1_vs_clinical,NaN,-0.0837,"[-0.165,-0.004]"
3,clinic,fusion_vs_clinical,NaN,0.0263,"[-0.012,0.064]"
4,clinic,fusion_vs_vol_t1,NaN,0.1099,"[0.055,0.165]"
5,combat_vs_nocombat,t1_d21_d32,NaN,-0.0040,mean ΔAUC over 5 fam; n_sig_fdr=0
6,combat_vs_nocombat,t1_only,NaN,-0.0092,mean ΔAUC over 5 fam; n_sig_fdr=0
7,soft_true_vs_false,t1_d21_d32,NaN,0.0217,descriptive mean Δ over 5 fam; n_pMCI 120 vs 74
8,soft_true_vs_false,t1_only,NaN,0.0214,descriptive mean Δ over 5 fam; n_pMCI 120 vs 74
